# 🛣️ Road Classification Production Notebook
## Pavement Type Classification for Road Survey Mapping

---

**Purpose:** Classify road surface types and generate a GPS-synced CSV for map visualization.

**Output:** `pavement_data.csv` with columns: `latitude`, `longitude`, `type`, `time`

## 1️⃣ Setup & GPU Check

In [ ]:
# @title 1️⃣ Setup & Configuration
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Enable GPU: Runtime → Change runtime type → GPU")

# @markdown ### ⚙️ Settings
FRAME_SKIP = 5 # @param {type:"integer"}
CONFIDENCE_THRESHOLD = 0.25
IMG_SIZE = 640

print(f"✅ Settings: FRAME_SKIP={FRAME_SKIP}")

!pip install -q ultralytics opencv-python pandas numpy tqdm

In [ ]:
# @title 2️⃣ Define File Paths
# @markdown Upload files to Colab first, then enter paths here.

# @markdown ### 📹 Input Video
VIDEO_PATH = "/content/video.mp4" # @param {type:"string"}

# @markdown ### 🧠 Classification Model
CLS_MODEL_PATH = "/content/best_cls.pt" # @param {type:"string"}

# @markdown ### 📍 GPS Log
GPS_LOG_PATH = "/content/gps_log.csv" # @param {type:"string"}

import os
for p, n in [(VIDEO_PATH, 'Video'), (CLS_MODEL_PATH, 'Model'), (GPS_LOG_PATH, 'GPS')]:
    if os.path.exists(p):
        print(f"✅ {n} found: {p}")
    else:
        print(f"⚠️ {n} NOT found: {p}")

## 3️⃣ Load GPS Data

In [ ]:
import pandas as pd

gps_data = None
video_start_time = None

if os.path.exists(GPS_LOG_PATH):
    try:
        df = pd.read_csv(GPS_LOG_PATH)
        df.columns = df.columns.str.strip().str.lower()
        
        # Find time column
        time_col = 'time' if 'time' in df.columns else 'timestamp'
        df[time_col] = pd.to_datetime(df[time_col], format='mixed', utc=True)
        df = df.sort_values(time_col)
        df = df.rename(columns={time_col: 'time'})
        
        gps_data = df
        video_start_time = gps_data['time'].iloc[0]
        print(f"✅ Loaded {len(gps_data)} GPS points")
    except Exception as e:
        print(f"❌ GPS Error: {e}")
else:
    print("❌ GPS file not found. Cannot proceed without GPS for road mapping.")

## 4️⃣ Load Classification Model

In [ ]:
from ultralytics import YOLO

cls_model = None
if os.path.exists(CLS_MODEL_PATH):
    cls_model = YOLO(CLS_MODEL_PATH)
    cls_model.to('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"✅ Model Loaded. Classes: {cls_model.names}")
else:
    print(f"❌ Model NOT found at {CLS_MODEL_PATH}")

## 5️⃣ Process Video (Classification Only)

In [ ]:
import cv2
from tqdm import tqdm

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print(f"❌ Error: Could not open video")
else:
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"✅ Video: {total_frames} frames @ {fps:.1f} FPS")

pavement_records = []
frame_idx = 0
pbar = tqdm(total=total_frames // FRAME_SKIP, desc="Classifying Road")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    if frame_idx % FRAME_SKIP == 0:
        # 1. Classification
        pavement_type = "Unknown"
        if cls_model:
            res = cls_model.predict(frame, verbose=False)[0]
            top1_idx = res.probs.top1
            pavement_type = res.names[top1_idx]
        
        # 2. GPS Sync
        lat, lon, timestamp = None, None, None
        if gps_data is not None and video_start_time is not None:
            video_time = video_start_time + pd.to_timedelta(frame_idx / fps, unit='s')
            closest_idx = (gps_data['time'] - video_time).abs().idxmin()
            gps_row = gps_data.loc[closest_idx]
            lat = float(gps_row['latitude'])
            lon = float(gps_row['longitude'])
            timestamp = str(gps_row['time'])
        
        # 3. Record
        if lat is not None and lon is not None:
            pavement_records.append({
                'latitude': lat,
                'longitude': lon,
                'type': pavement_type,
                'time': timestamp
            })
        
        pbar.update(1)
    
    frame_idx += 1

cap.release()
pbar.close()
print(f"\n✅ Complete. Collected {len(pavement_records)} road type samples.")

## 6️⃣ Save & Download Results

In [ ]:
from google.colab import files
import os

os.makedirs('results', exist_ok=True)

if len(pavement_records) > 0:
    df = pd.DataFrame(pavement_records)
    df.to_csv('results/pavement_data.csv', index=False)
    
    print("📊 Pavement Type Breakdown:")
    print(df['type'].value_counts())
    
    files.download('results/pavement_data.csv')
    print("🎉 Downloaded pavement_data.csv")
else:
    print("⚠️ No records to save.")